<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from google.colab import drive
from xgboost import XGBRegressor

# ============================================================
# 0. xgboost 설치 (처음 1번만)
# ============================================================
# !pip install xgboost

# ============================================================
# 1. 데이터 로드
# ============================================================
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx'
df_raw = pd.read_excel(file_path)

# ============================================================
# 2. 전처리 함수
# ============================================================
def prepare_seoul_df_final(df_input):
    df = df_input.copy()

    # Size
    if "Size_m2" in df.columns:
        df["Size_Level"] = df["Size_m2"]

    # Population Density
    if "Pop. Density" in df.columns:
        df["Pop_Density_Level"] = df["Pop. Density"]

    # Units
    if "num_of_people" in df.columns:
        df["Units_Level"] = df["num_of_people"]

    # 비율 변수
    age_map = {
        "Median age": "Medium_age_ratio",
        "Old Population": "Old_pop_ratio",
        "Sex_ratio": "Sex_ratio_ratio"
    }

    for original, new in age_map.items():
        if original in df.columns:
            df[new] = df[original] / 100.0 if df[original].max() > 2.0 else df[original]

    # CBD 거리 km화
    if "Dist_CBD" in df.columns:
        df["Dist_CBD_km"] = df["Dist_CBD"] / 1000.0

    # 계절 더미
    if "Month_Sold" in df.columns:
        m = pd.to_numeric(df["Month_Sold"], errors="coerce")
        df["Spring"] = m.isin([3, 4, 5]).astype(int)
        df["Fall"]   = m.isin([9, 10, 11]).astype(int)
        df["Winter"] = m.isin([12, 1, 2]).astype(int)

    return df

# ============================================================
# 3. XGBoost 실행 함수
# ============================================================
def run_xgboost_prediction(df_sub, feature_cols, target_col="Log_Price_per_m2", random_state=42):
    """
    XGBoost 회귀:
    - Train / Test 성능 비교
    - Test 예측값 저장
    """

    df_model = df_sub.copy()

    # 사용할 열만 남기고 결측 제거
    use_cols = [c for c in feature_cols + [target_col] if c in df_model.columns]
    df_model = df_model.dropna(subset=use_cols).reset_index(drop=True)

    X = df_model[[c for c in feature_cols if c in df_model.columns]]
    y = df_model[target_col]

    # train / test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=random_state
    )

    # 모델 생성
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective='reg:squarederror',
        random_state=random_state,
        n_jobs=-1
    )

    # 학습
    xgb.fit(X_train, y_train)

    # ----------------------------
    # Train 성능
    # ----------------------------
    y_pred_train = xgb.predict(X_train)
    r2_train = r2_score(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))

    # ----------------------------
    # Test 성능
    # ----------------------------
    y_pred_test = xgb.predict(X_test)
    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    # Test 예측값 테이블
    pred_df = pd.DataFrame({
        "Actual": y_test.values,
        "Predicted": y_pred_test
    }).reset_index(drop=True)

    return {
        "model": xgb,
        "r2_train": r2_train,
        "rmse_train": rmse_train,
        "r2_test": r2_test,
        "rmse_test": rmse_test,
        "pred_df": pred_df,
        "n_train": len(X_train),
        "n_test": len(X_test)
    }

# ============================================================
# 4. 데이터 준비
# ============================================================
df_all = prepare_seoul_df_final(df_raw)

# 사용할 변수
features = [
    "Size_Level",
    "Floor",
    "Units_Level",
    "Parking_per_Household",
    "Construction_Year",
    "Log_Dist_Subway",
    "Log_Dist_Green",
    "Log_Dist_Water",
    "Dist_CBD_km",
    "Sex_ratio_ratio",
    "Pop_Density_Level",
    "Medium_age_ratio",
    "Old_pop_ratio",
    "Spring",
    "Fall",
    "Winter"
]

# ============================================================
# 5. 연도별 XGBoost
# ============================================================
xgb_results_by_year = []
xgb_prediction_dict = {}

for year in [2022, 2023]:
    df_year = df_all[df_all["Year_Sold"] == year].copy()

    valid_cols = [c for c in features + ["Log_Price_per_m2"] if c in df_year.columns]
    df_year = df_year.dropna(subset=valid_cols)

    if len(df_year) < 30:
        print(f"{year}: 표본 수 부족으로 건너뜀")
        continue

    xgb_out = run_xgboost_prediction(
        df_sub=df_year,
        feature_cols=features,
        target_col="Log_Price_per_m2",
        random_state=42
    )

    xgb_results_by_year.append({
        "Year": year,
        "Train_R2": xgb_out["r2_train"],
        "Train_RMSE": xgb_out["rmse_train"],
        "Test_R2": xgb_out["r2_test"],
        "Test_RMSE": xgb_out["rmse_test"],
        "Train_N": xgb_out["n_train"],
        "Test_N": xgb_out["n_test"]
    })

    xgb_prediction_dict[year] = xgb_out["pred_df"]

    print(f"\n[{year} XGBoost 결과]")
    print("Train 성능")
    print(f"R2   : {xgb_out['r2_train']:.4f}")
    print(f"RMSE : {xgb_out['rmse_train']:.4f}")

    print("Test 성능")
    print(f"R2   : {xgb_out['r2_test']:.4f}")
    print(f"RMSE : {xgb_out['rmse_test']:.4f}")

    print(f"Train N : {xgb_out['n_train']}")
    print(f"Test N  : {xgb_out['n_test']}")

# ============================================================
# 6. 전체기간 XGBoost
# ============================================================
df_full = df_all.copy()

valid_cols_full = [c for c in features + ["Log_Price_per_m2"] if c in df_full.columns]
df_full = df_full.dropna(subset=valid_cols_full)

xgb_full_out = run_xgboost_prediction(
    df_sub=df_full,
    feature_cols=features,
    target_col="Log_Price_per_m2",
    random_state=42
)

print("\n[전체기간 XGBoost 결과]")
print("Train 성능")
print(f"R2   : {xgb_full_out['r2_train']:.4f}")
print(f"RMSE : {xgb_full_out['rmse_train']:.4f}")

print("Test 성능")
print(f"R2   : {xgb_full_out['r2_test']:.4f}")
print(f"RMSE : {xgb_full_out['rmse_test']:.4f}")

print(f"Train N : {xgb_full_out['n_train']}")
print(f"Test N  : {xgb_full_out['n_test']}")

# ============================================================
# 7. 성능표 정리
# ============================================================
xgb_results_table = pd.DataFrame(xgb_results_by_year)

full_row = pd.DataFrame([{
    "Year": "Full Sample",
    "Train_R2": xgb_full_out["r2_train"],
    "Train_RMSE": xgb_full_out["rmse_train"],
    "Test_R2": xgb_full_out["r2_test"],
    "Test_RMSE": xgb_full_out["rmse_test"],
    "Train_N": xgb_full_out["n_train"],
    "Test_N": xgb_full_out["n_test"]
}])

xgb_results_table = pd.concat([xgb_results_table, full_row], ignore_index=True)

print("\n[XGBoost 성능 비교표]")
display(xgb_results_table)

# ============================================================
# 8. 예측값 일부 확인
# ============================================================
for year, pred_df in xgb_prediction_dict.items():
    print(f"\n[{year} 예측값 일부]")
    display(pred_df.head())

print("\n[전체기간 예측값 일부]")
display(xgb_full_out["pred_df"].head())

# ============================================================
# 9. 엑셀 저장
# ============================================================
output_path = '/content/drive/MyDrive/Seoul_Apartment_XGB_Results.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # 성능표 저장
    xgb_results_table.to_excel(writer, sheet_name='XGB_Performance', index=False)

    # 연도별 예측값 저장
    for year, pred_df in xgb_prediction_dict.items():
        pred_df.to_excel(writer, sheet_name=f'Pred_{year}', index=False)

    # 전체기간 예측값 저장
    xgb_full_out["pred_df"].to_excel(writer, sheet_name='Pred_Full', index=False)

print(f"\n저장 완료: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[2022 XGBoost 결과]
Train 성능
R2   : 0.9512
RMSE : 0.0994
Test 성능
R2   : 0.8845
RMSE : 0.1556
Train N : 7405
Test N  : 3174

[2023 XGBoost 결과]
Train 성능
R2   : 0.9514
RMSE : 0.0951
Test 성능
R2   : 0.9302
RMSE : 0.1145
Train N : 21749
Test N  : 9322

[전체기간 XGBoost 결과]
Train 성능
R2   : 0.9181
RMSE : 0.1326
Test 성능
R2   : 0.9094
RMSE : 0.1385
Train N : 113662
Test N  : 48713

[XGBoost 성능 비교표]


,Year,Train_R2,Train_RMSE,Test_R2,Test_RMSE,Train_N,Test_N
0,2022,0.951155,0.099410,0.884531,0.155567,7405,3174
1,2023,0.951411,0.095144,0.930219,0.114492,21749,9322
2,Full Sample,0.918056,0.132581,0.909378,0.138497,113662,48713



[2022 예측값 일부]


,Actual,Predicted
0,17.083235,16.977709
1,15.850185,16.233612
2,16.951378,16.822836
3,16.239966,16.282166
4,16.471918,16.268473



[2023 예측값 일부]


,Actual,Predicted
0,16.454920,16.415052
1,16.102828,16.155270
2,16.659107,16.629004
3,16.075036,16.151585
4,16.046305,16.042063



[전체기간 예측값 일부]


,Actual,Predicted
0,16.463642,16.468622
1,16.152554,16.194782
2,16.483932,16.144829
3,16.355151,16.230413
4,16.445963,16.525019



저장 완료: /content/drive/MyDrive/Seoul_Apartment_XGB_Results.xlsx
